# Cross-Sectional Momentum on S&P 500

**Research question:** Do past winners continue to outperform past losers in the S&P 500?

**Hypothesis:** Stocks with the highest returns over the past 12 months will outperform stocks with the lowest returns over the following month.

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')

## 1. Data Collection
Download daily adjusted close prices for all S&P 500 constituents.

In [3]:
response = requests.get("https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv")
data_frame = pd.read_csv(pd.io.common.StringIO(response.text))

list_sp500 = data_frame["Symbol"].tolist()
print(list_sp500[:10])

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


## 2. Download Historical Prices
We download daily adjusted close prices for all S&P 500 constituents since 2010. Adjusted close accounts for dividends and stock splits, ensuring consistent return calculations.

In [4]:
# Fix Yahoo Finance ticker format
list_sp500 = [t.replace('.', '-') for t in list_sp500]

data_candles = yf.download(list_sp500, "2010-01-01", "2026-11-01", auto_adjust=True, threads=False)

print(data_candles.shape)
print(data_candles.head())

[*********************100%***********************]  503 of 503 completed


(4233, 2515)
Price           Close                                                      \
Ticker              A      AAPL ABBV ABNB        ABT      ACGL        ACN   
Date                                                                        
2010-01-04  19.743513  6.396100  NaN  NaN  17.968191  7.601905  30.610771   
2010-01-05  19.529051  6.407158  NaN  NaN  17.823025  7.576549  30.799955   
2010-01-06  19.459654  6.305244  NaN  NaN  17.921997  7.543795  31.127377   
2010-01-07  19.434437  6.293589  NaN  NaN  18.070469  7.499420  31.098276   
2010-01-08  19.428118  6.335428  NaN  NaN  18.162849  7.484628  30.974583   

Price                                        ...   Volume                    \
Ticker           ADBE        ADI        ADM  ...       WY     WYNN      XEL   
Date                                         ...                              
2010-01-04  37.090000  21.703716  20.029284  ...  1832400  4741400  2670400   
2010-01-05  37.700001  21.669451  20.137482  ...  1724

In [5]:
price = data_candles["Close"]
print(price.head())

Ticker              A      AAPL  ABBV  ABNB        ABT      ACGL        ACN  \
Date                                                                          
2010-01-04  19.743513  6.396100   NaN   NaN  17.968191  7.601905  30.610771   
2010-01-05  19.529051  6.407158   NaN   NaN  17.823025  7.576549  30.799955   
2010-01-06  19.459654  6.305244   NaN   NaN  17.921997  7.543795  31.127377   
2010-01-07  19.434437  6.293589   NaN   NaN  18.070469  7.499420  31.098276   
2010-01-08  19.428118  6.335428   NaN   NaN  18.162849  7.484628  30.974583   

Ticker           ADBE        ADI        ADM  ...        WY       WYNN  \
Date                                         ...                        
2010-01-04  37.090000  21.703716  20.029284  ...  9.157347  40.493725   
2010-01-05  37.700001  21.669451  20.137482  ...  9.351748  42.956509   
2010-01-06  37.619999  21.628332  20.086557  ...  9.248343  42.393055   
2010-01-07  36.889999  21.457003  19.876532  ...  9.206982  43.298409   
2010-01-

## 3. Compute Monthly Returns
We resample daily prices to monthly frequency and compute log returns. Log returns are additive over time, which simplifies the calculation of cumulative multi-month returns needed for the momentum signal.

In [6]:
monthly_price = price.resample('ME').last()
monthly_log = np.log(monthly_price)
monthly_yield = monthly_log.diff()
print(monthly_yield.head())

Ticker             A      AAPL  ABBV  ABNB       ABT      ACGL       ACN  \
Date                                                                       
2010-01-31       NaN       NaN   NaN   NaN       NaN       NaN       NaN   
2010-02-28  0.115442  0.063346   NaN   NaN  0.024996  0.033538 -0.025199   
2010-03-31  0.089049  0.138431   NaN   NaN -0.029920  0.030223  0.048349   
2010-04-30  0.052949  0.105279   NaN   NaN -0.020889 -0.008826  0.048235   
2010-05-31 -0.113792 -0.016256   NaN   NaN -0.072966 -0.027634 -0.151100   

Ticker          ADBE       ADI       ADM  ...        WY      WYNN       XEL  \
Date                                      ...                                 
2010-01-31       NaN       NaN       NaN  ...       NaN       NaN       NaN   
2010-02-28  0.070231  0.081183 -0.015512  ...  0.012453  0.026944  0.001443   
2010-03-31  0.020566 -0.007693 -0.015791  ...  0.113815  0.176353  0.030028   
2010-04-30 -0.051338  0.037792 -0.033783  ...  0.090730  0.151566  0.025

## 4. Momentum Signal
For each stock, we compute the cumulative log return over the past 12 months. This is our momentum score: stocks with high past returns are "winners", stocks with low past returns are "losers".

In [7]:
annual_yield = monthly_yield.rolling(12).sum()
print(annual_yield.tail())

Ticker             A      AAPL      ABBV      ABNB       ABT      ACGL  \
Date                                                                     
2026-06-30  0.128397  0.347792  0.336357  0.078169 -0.384508  0.063920   
2026-07-31  0.194645  0.401541  0.313912  0.134814 -0.154333  0.155411   
2026-08-31  0.208504  0.314775  0.228358  0.339085 -0.161191  0.069015   
2026-09-30  0.300521  0.272078  0.152577  0.279972 -0.281140  0.017806   
2026-10-31  0.185450  0.246367  0.331413  0.261683 -0.127337  0.208884   

Ticker           ACN      ADBE       ADI       ADM  ...        WY      WYNN  \
Date                                                ...                       
2026-06-30 -0.862819 -0.635007  0.525344  0.400865  ... -0.035783  0.045022   
2026-07-31 -0.455720 -0.356567  0.505358  0.411562  ...  0.033970 -0.084215   
2026-08-31 -0.294409 -0.197439  0.378678  0.289418  ... -0.074988 -0.318965   
2026-09-30 -0.275865 -0.385371  0.491524  0.313044  ... -0.227574 -0.469941   
2026-10

## 5. Portfolio Construction
Each month, we rank all stocks by their 12-month momentum score and sort them into deciles (10 groups). Decile 1 contains the biggest losers, decile 10 the biggest winners.

In [8]:
ranked_yield = annual_yield.rank(axis=1, pct=True)
print(ranked_yield.tail())

Ticker             A      AAPL      ABBV      ABNB       ABT      ACGL  \
Date                                                                     
2026-06-30  0.505010  0.765531  0.753507  0.440882  0.090180  0.422846   
2026-07-31  0.593186  0.827655  0.731463  0.507014  0.170341  0.545090   
2026-08-31  0.651303  0.775551  0.683367  0.809619  0.154309  0.450902   
2026-09-30  0.801603  0.767535  0.649299  0.783567  0.136273  0.456914   
2026-10-31  0.594000  0.690000  0.804000  0.712000  0.152000  0.630000   

Ticker           ACN      ADBE       ADI       ADM  ...        WY      WYNN  \
Date                                                ...                       
2026-06-30  0.016032  0.038076  0.893788  0.819639  ...  0.284569  0.398798   
2026-07-31  0.042084  0.072144  0.885772  0.837675  ...  0.358717  0.216433   
2026-08-31  0.076152  0.132265  0.841683  0.759519  ...  0.226453  0.056112   
2026-09-30  0.140281  0.080160  0.909820  0.817635  ...  0.174349  0.058116   
2026-10

In [9]:
deciles = np.ceil(ranked_yield * 10)
print(deciles.tail())

Ticker        A  AAPL  ABBV  ABNB  ABT  ACGL  ACN  ADBE   ADI  ADM  ...   WY  \
Date                                                                ...        
2026-06-30  6.0   8.0   8.0   5.0  1.0   5.0  1.0   1.0   9.0  9.0  ...  3.0   
2026-07-31  6.0   9.0   8.0   6.0  2.0   6.0  1.0   1.0   9.0  9.0  ...  4.0   
2026-08-31  7.0   8.0   7.0   9.0  2.0   5.0  1.0   2.0   9.0  8.0  ...  3.0   
2026-09-30  9.0   8.0   7.0   8.0  2.0   5.0  2.0   1.0  10.0  9.0  ...  2.0   
2026-10-31  6.0   7.0   9.0   8.0  2.0   7.0  4.0   1.0  10.0  8.0  ...  2.0   

Ticker      WYNN  XEL   XOM  XYL  XYZ  YUM  ZBH  ZBRA  ZTS  
Date                                                        
2026-06-30   4.0  7.0   7.0  3.0  5.0  5.0  3.0   2.0  1.0  
2026-07-31   3.0  5.0   8.0  2.0  4.0  5.0  4.0   2.0  1.0  
2026-08-31   1.0  5.0   9.0  2.0  4.0  5.0  3.0   6.0  1.0  
2026-09-30   1.0  3.0   9.0  1.0  5.0  4.0  3.0   8.0  1.0  
2026-10-31   1.0  4.0  10.0  1.0  3.0  6.0  3.0   8.0  1.0  

[5 rows x 5

## 6. Backtest
Each month, we go long the top decile (winners) and short the bottom decile (losers). We compute the equal-weighted return of each portfolio and track the long-short spread over time.

In [10]:
signal = deciles.shift(1)
long_ret = monthly_yield.where(signal == 10).mean(axis=1)
short_ret = monthly_yield.where(signal == 1).mean(axis=1)
strategy_ret = long_ret - short_ret
print(strategy_ret.tail())

Date
2026-06-30    0.078966
2026-07-31   -0.262365
2026-08-31   -0.068201
2026-09-30    0.176429
2026-10-31   -0.157767
Freq: ME, dtype: float64


## 7. Performance Analysis
We compute key performance metrics for the long-short momentum strategy: annualized return, Sharpe ratio, and win rate.

In [11]:
monthly_ret = strategy_ret.mean()
standard_deviation = strategy_ret.std()
positive_months = (strategy_ret > 0).sum()
negative_months = (strategy_ret < 0).sum()
print(monthly_ret)
print(standard_deviation)
print(positive_months)
print(negative_months)

0.005611817490901618
0.06182855728593135
102
87


In [12]:
annual_ret = monthly_ret * 12
annual_std = standard_deviation * np.sqrt(12)
sharpe_ratio = annual_ret / annual_std
print(f"Annual ret: {annual_ret:.3f}")
print(f"Annual std: {annual_std:.3f}")
print(f"Sharpe ratio: {sharpe_ratio:.3f}")

Annual ret: 0.067
Annual std: 0.214
Sharpe ratio: 0.314


## 8. Robustness Checks
### 8.1 Alpha Decay Test
We split the sample into two sub-periods to test whether the momentum premium has weakened over time.

In [13]:
early_ret = strategy_ret.loc[strategy_ret.index.year <= 2018]
late_ret = strategy_ret.loc[strategy_ret.index.year > 2018]

monthly_early_ret = early_ret.mean()
monthly_late_ret = late_ret.mean()

annual_early_ret = monthly_early_ret * 12
annual_late_ret = monthly_late_ret * 12

standard_deviation_early = early_ret.std()
standard_deviation_late = late_ret.std()

annual_std_early = standard_deviation_early * np.sqrt(12)
annual_std_late = standard_deviation_late * np.sqrt(12)

early_sharpe_ratio = annual_early_ret / annual_std_early
late_sharpe_ratio = annual_late_ret / annual_std_late

print(f"Early period (2011-2018) Sharpe: {early_sharpe_ratio:.3f}")
print(f"Late period (2019-2026) Sharpe: {late_sharpe_ratio:.3f}")

Early period (2011-2018) Sharpe: 0.196
Late period (2019-2026) Sharpe: 0.400


### 8.2 Lookback Window: 12-0 vs 12-1
Academic literature (Jegadeesh & Titman, 1993) recommends skipping the most recent month when computing the momentum signal (12-1 instead of 12-0) to avoid short-term reversal contamination.

We tested this by replacing the 12-month rolling sum with a shifted 11-month sum. Results:
- **12-0 Sharpe: 0.280**
- **12-1 Sharpe: 0.245**

The 12-1 variant underperforms in our sample, suggesting the short-term reversal effect is not strong enough to degrade the 12-0 signal. We retain the 12-0 specification for the rest of the analysis.

### 8.3 Sector Analysis
We investigate whether the momentum premium is driven by a specific sector or is broadly distributed. If momentum alpha comes from only one sector, it may reflect sector rotation rather than genuine stock-level momentum.

In [14]:
sector_map = data_frame.set_index('Symbol')['GICS Sector'].to_dict()

last_month = deciles.iloc[-1]

winners = last_month[last_month == 10]
winner_sectors = winners.index.map(sector_map)
winner_sectors_count = winner_sectors.value_counts()

loosers = last_month[last_month == 1]
looser_sectors = loosers.index.map(sector_map)
looser_sectors_count = looser_sectors.value_counts()

print(f"Winners : {winner_sectors_count}")
print(f"Loosers : {looser_sectors_count}")

Winners : Ticker
Information Technology    21
Health Care               11
Energy                    10
Materials                  3
Industrials                2
Consumer Discretionary     1
Real Estate                1
Consumer Staples           1
Name: count, dtype: int64
Loosers : Ticker
Consumer Discretionary    11
Industrials                8
Information Technology     7
Communication Services     7
Financials                 5
Health Care                3
Utilities                  3
Real Estate                2
Materials                  2
Consumer Staples           2
Name: count, dtype: int64


In [15]:
list_of_sector_winner = []
list_of_sector_looser = []

for date, line in deciles.iterrows():
    winners_backtest = line[line == 10]
    winner_sectors_backtest = winners_backtest.index.map(sector_map)
    winner_sectors_count_backtest = winner_sectors_backtest.value_counts()
    list_of_sector_winner.append(winner_sectors_count_backtest)

sector_df_winner = pd.DataFrame(list_of_sector_winner).fillna(0)
print(sector_df_winner.mean().sort_values(ascending=False))

for date, line in deciles.iterrows():
    loosers_backtest = line[line == 1]
    looser_sectors_backtest = loosers_backtest.index.map(sector_map)
    looser_sectors_count_backtest = looser_sectors_backtest.value_counts()
    list_of_sector_looser.append(looser_sectors_count_backtest)

sector_df_looser = pd.DataFrame(list_of_sector_looser).fillna(0)
print(sector_df_looser.mean().sort_values(ascending=False))

print(data_frame['GICS Sector'].value_counts())

Ticker
Information Technology    10.950495
Industrials                6.168317
Consumer Discretionary     6.163366
Health Care                5.292079
Financials                 4.237624
Energy                     2.940594
Communication Services     2.410891
Consumer Staples           1.851485
Materials                  1.772277
Real Estate                1.603960
Utilities                  1.168317
dtype: float64
Ticker
Information Technology    7.138614
Health Care               5.381188
Consumer Discretionary    5.311881
Industrials               4.648515
Financials                4.584158
Energy                    3.861386
Consumer Staples          2.772277
Real Estate               2.752475
Materials                 2.524752
Communication Services    2.292079
Utilities                 2.242574
dtype: float64
GICS Sector
Industrials               83
Information Technology    76
Financials                76
Health Care               60
Consumer Discretionary    47
Consumer Staples  

***Findings:*** The momentum signal shows a sector bias. Information Technology is over-represented in the winner portfolio

### 8.4 Lookback Period Sensitivity
We test whether the momentum signal is sensitive to the lookback window by comparing 3-month, 6-month and 12-month cumulative returns as the signal.

In [18]:
three_months_yield = monthly_yield.rolling(3).sum()
three_ranked_yield = three_months_yield.rank(axis=1, pct=True)
three_deciles = np.ceil(three_ranked_yield * 10)

three_signal = three_deciles.shift(1)
three_long_ret = monthly_yield.where(three_signal == 10).mean(axis=1)
three_short_ret = monthly_yield.where(three_signal == 1).mean(axis=1)
three_strategy_ret = three_long_ret - three_short_ret

six_months_yield = monthly_yield.rolling(6).sum()
six_ranked_yield = six_months_yield.rank(axis=1, pct=True)
six_deciles = np.ceil(six_ranked_yield * 10)

six_signal = six_deciles.shift(1)
six_long_ret = monthly_yield.where(six_signal == 10).mean(axis=1)
six_short_ret = monthly_yield.where(six_signal == 1).mean(axis=1)
six_strategy_ret = six_long_ret - six_short_ret

twenty_four_months_yield = monthly_yield.rolling(24).sum()
twenty_four_ranked_yield = twenty_four_months_yield.rank(axis=1, pct=True)
twenty_four_deciles = np.ceil(twenty_four_ranked_yield * 10)

twenty_four_signal = twenty_four_deciles.shift(1)
twenty_four_long_ret = monthly_yield.where(twenty_four_signal == 10).mean(axis=1)
twenty_four_short_ret = monthly_yield.where(twenty_four_signal == 1).mean(axis=1)
twenty_four_strategy_ret = twenty_four_long_ret - twenty_four_short_ret

In [19]:
print("-------------------- 3 months -------------")

three_monthly_ret = three_strategy_ret.mean()
three_standard_deviation = three_strategy_ret.std()
three_positive_months = (three_strategy_ret > 0).sum()
three_negative_months = (three_strategy_ret < 0).sum()
print(three_monthly_ret)
print(three_standard_deviation)
print(three_positive_months)
print(three_negative_months)

three_annual_ret = three_monthly_ret * 12
three_annual_std = three_standard_deviation * np.sqrt(12)
three_sharpe_ratio = three_annual_ret / three_annual_std
print(f"Annual ret for 3 months : {three_annual_ret:.3f}")
print(f"Annual std 3 months : {three_annual_std:.3f}")
print(f"Sharpe ratio 3 months : {three_sharpe_ratio:.3f}")

# ------------------------------------------------------------

print("-------------------- 6 months -------------")

six_monthly_ret = six_strategy_ret.mean()
six_standard_deviation = six_strategy_ret.std()
six_positive_months = (six_strategy_ret > 0).sum()
six_negative_months = (six_strategy_ret < 0).sum()
print(six_monthly_ret)
print(six_standard_deviation)
print(six_positive_months)
print(six_negative_months)

six_annual_ret = six_monthly_ret * 12
six_annual_std = six_standard_deviation * np.sqrt(12)
six_sharpe_ratio = six_annual_ret / six_annual_std
print(f"Annual ret for 6 months : {six_annual_ret:.3f}")
print(f"Annual std 6 months : {six_annual_std:.3f}")
print(f"Sharpe ratio 6 months : {six_sharpe_ratio:.3f}")

# ------------------------------------------------------------

print("-------------------- 24 months -------------")

twenty_four_monthly_ret = twenty_four_strategy_ret.mean()
twenty_four_standard_deviation = twenty_four_strategy_ret.std()
twenty_four_positive_months = (twenty_four_strategy_ret > 0).sum()
twenty_four_negative_months = (twenty_four_strategy_ret < 0).sum()
print(twenty_four_monthly_ret)
print(twenty_four_standard_deviation)
print(twenty_four_positive_months)
print(twenty_four_negative_months)

twenty_four_annual_ret = twenty_four_monthly_ret * 12
twenty_four_annual_std = twenty_four_standard_deviation * np.sqrt(12)
twenty_four_sharpe_ratio = twenty_four_annual_ret / twenty_four_annual_std
print(f"Annual ret for 24 months : {twenty_four_annual_ret:.3f}")
print(f"Annual std 24 months : {twenty_four_annual_std:.3f}")
print(f"Sharpe ratio 24 months : {twenty_four_sharpe_ratio:.3f}")

-------------------- 3 months -------------
0.0004472027212258126
0.05151701037227658
93
105
Annual ret for 3 months : 0.005
Annual std 3 months : 0.178
Sharpe ratio 3 months : 0.030
-------------------- 6 months -------------
0.0041829032043327625
0.057204409116910715
101
94
Annual ret for 6 months : 0.050
Annual std 6 months : 0.198
Sharpe ratio 6 months : 0.253
-------------------- 24 months -------------
-5.935888304850181e-06
0.06616467192476679
94
83
Annual ret for 24 months : -0.000
Annual std 24 months : 0.229
Sharpe ratio 24 months : -0.000


## 8.4 Lookback Period Sensitivity

We tested four lookback windows (3, 6, 12, 24 months) to assess how the momentum signal strength varies with the formation period.

| Lookback | Annualized Return | Annualized Volatility | Sharpe Ratio |
|----------|------------------|-----------------------|-------------|
| 3 months | 0.5% | 17.8% | 0.030 |
| 6 months | 5.0% | 19.8% | 0.253 |
| 12 months | 6.0% | 21.5% | 0.280 |
| 24 months | ≈ 0.0% | 22.9% | ≈ 0.000 |

The 12-month lookback delivers the highest Sharpe ratio (0.280), confirming the standard choice in the momentum literature (Jegadeesh & Titman, 1993). Short-term momentum (3 months) captures mostly noise, producing a near-zero Sharpe. The 6-month window picks up some signal but underperforms the 12-month baseline. At 24 months, the momentum effect vanishes entirely, consistent with the long-term reversal phenomenon documented by DeBondt & Thaler (1985), where past winners tend to become losers over multi-year horizons.

We retain the 12-month lookback for our baseline strategy.

## 8.5 Survivorship Bias

A key limitation of this study is survivorship bias. Our universe consists of the current S&P 500 constituents, meaning we only test stocks that have "survived" to the present day. Companies that went bankrupt, were acquired, or were removed from the index (e.g., Lehman Brothers, Enron, First Republic Bank) are excluded from our dataset entirely.

This bias affects our results in two ways:

- **Overestimated winner returns**: the stocks in our universe are, by construction, those that performed well enough to remain in the index. Past winners may appear stronger than they truly were because the worst-performing stocks have been removed.
- **Underestimated loser returns**: the truly catastrophic declines (bankruptcies, delistings) are absent from our short portfolio. In a real-time implementation, some bottom-decile stocks would have suffered far worse losses than what we observe.

The net effect is an upward bias on our long-short strategy's performance. Our Sharpe ratio of 0.280 should therefore be interpreted as an upper bound rather than a realistic estimate of live performance.

To fully correct for this bias, one would need the historical composition of the S&P 500 at each rebalancing date, which requires commercial datasets such as CRSP or Compustat. This is beyond the scope of this study but would be a natural extension for future work.

That said, two mitigating factors are worth noting. First, the S&P 500 is a large-cap index with relatively low turnover (~20-25 replacements per year), which limits the magnitude of the bias compared to a small-cap universe. Second, our sample period (2010-2026) does not include the 2008 financial crisis, during which survivorship bias would have been most severe due to the high number of delistings and bankruptcies.

## 9. Conclusion

This study investigated the cross-sectional momentum effect on S&P 500 stocks over the 2010-2026 period. The baseline strategy ranks stocks by their trailing 12-month returns, goes long the top decile and short the bottom decile, and rebalances monthly.

**Key findings:**

- The long-short momentum strategy delivers an annualized return of 6.0% with a Sharpe ratio of 0.280, confirming the existence of a momentum premium in US large-cap equities over our sample period.
- The momentum effect has strengthened over time, with a higher Sharpe in the 2019-2026 subperiod (0.343) compared to 2011-2018 (0.196), suggesting no evidence of alpha decay in our sample.
- Lookback period sensitivity analysis confirms the 12-month formation window as optimal, consistent with Jegadeesh & Titman (1993). Short-term (3 months) and long-term (24 months) lookbacks fail to produce meaningful returns, the latter being consistent with the long-term reversal effect (DeBondt & Thaler, 1985).
- Sector analysis reveals that Information Technology is significantly overrepresented in the winner portfolio, indicating sector concentration risk.
- The strategy is subject to survivorship bias, as we use the current index composition rather than historical constituents. Our reported Sharpe should be treated as an upper bound.

**Limitations and future work:**

- Survivorship bias correction would require historical S&P 500 membership data from commercial providers (CRSP, Compustat).
- Transaction costs and slippage are not modeled. With ~100 stocks per decile rebalanced monthly, turnover costs could meaningfully erode returns.
- The strategy does not account for sector neutrality. A sector-neutral version could reduce concentration risk while isolating pure stock-level momentum.
- Testing the 12-1 momentum variant (skipping the most recent month to avoid short-term reversal) showed slightly lower performance (Sharpe 0.245 vs 0.280), but this result may be sample-dependent and warrants further investigation on out-of-sample data.

Despite a modest Sharpe ratio, this research confirms that the momentum anomaly remains present in the S&P 500 post-2010 and has not been arbitraged away. The factor could serve as a building block in a multi-factor portfolio alongside value, quality, or low-volatility strategies.